# 21 — Schema Cleaning, Dynamic Concatenation, & Alignment Integrity
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to schema reconciliation, programmatic column pruning, continuous index generation, and series alignment in Pandas.*

---

## 📌 Executive Summary & Interview Expectations
In production ingestion pipelines (e.g. ingesting batches of CSVs from IoT sensors or external vendors), datasets frequently suffer from **schema drift**: trailing empty delimiter columns (`Unnamed: X`), split file partitions, and non-aligned indices.
Interviewers use this scenario to evaluate whether you write brittle hardcoded fixes or robust, production-grade transformations:
1. **Dynamic Column Pruning**: Programmatically removing ghost/empty delimiter columns instead of hardcoding column names.
2. **Schema Alignment & Concatenation**: Safely stacking DataFrames with matching vs disjoint schemas using `pd.concat`.
3. **Index Collision Avoidance**: Understanding why `ignore_index=True` is mandatory when stacking partitioned files to prevent duplicate label lookups.
4. **Series-to-DataFrame Assignment Mechanics**: How Pandas aligns indices during column assignment and how index mismatches introduce silent `NaN` values.

## 1. Environment Setup & Data Ingestion

In [1]:
import numpy as np
import pandas as pd

# Load partitioned vehicle datasets
cars1 = pd.read_csv("cars1.csv")
cars2 = pd.read_csv("cars2.csv")

print(f"Cars 1 Initial Shape: {cars1.shape}")
print(f"Cars 2 Initial Shape: {cars2.shape}")
cars1.head(3)

Cars 1 Initial Shape: (198, 14)
Cars 2 Initial Shape: (200, 9)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model,origin,car,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,18.0,8,307,130,3504,12.0,70,1,chevrolet chevelle malibu,NaN,NaN,NaN,NaN,NaN
1,15.0,8,350,165,3693,11.5,70,1,buick skylark 320,NaN,NaN,NaN,NaN,NaN
2,18.0,8,318,150,3436,11.0,70,1,plymouth satellite,NaN,NaN,NaN,NaN,NaN


## 2. Programmatic Column Pruning: Beyond Brittle Hardcoding

### ⚠️ Junior Anti-Pattern vs Senior Production Standard
```python
# Junior Brittle Approach (Breaks if column count or naming changes):
cars1.drop(columns=['Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13'], inplace=True)

# Senior Dynamic Approach 1 (Drop all columns that are 100% null):
cars1 = cars1.dropna(axis=1, how='all')

# Senior Dynamic Approach 2 (Filter out Unnamed delimiter columns via regex/string):
cars1 = cars1.loc[:, ~cars1.columns.str.startswith('Unnamed')]
```

In [2]:
# Inspect columns of cars1
print("Original cars1 columns:", cars1.columns.tolist())

# Robust dynamic column pruning: remove all columns where all values are NaN
cars1 = cars1.dropna(axis=1, how="all")

print("\nPruned cars1 columns:", cars1.columns.tolist())
print(f"Verified matching column count: {cars1.shape[1] == cars2.shape[1]} (Both have {cars1.shape[1]} columns)")

Original cars1 columns: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model', 'origin', 'car', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Pruned cars1 columns: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model', 'origin', 'car']
Verified matching column count: True (Both have 9 columns)


## 3. Concatenation and the Duplicate Index Trap

### 🚨 Top Interview Gotcha: The Silent Duplicate Index Collision
- If you concatenate two DataFrames of length 198 and 200 without `ignore_index=True`, both share indices $0..197$.
- If you subsequently call `cars.loc[0]`, Pandas returns **2 rows** instead of 1 scalar record!
- Downstream merges, series assignments, and lookups can fail or silently multiply data!

In [3]:
# Row counts
n1, n2 = cars1.shape[0], cars2.shape[0]
print(f"cars1 rows: {n1} | cars2 rows: {n2} | Expected total: {n1 + n2}")

# Concatenating with continuous 0..N-1 index
cars = pd.concat([cars1, cars2], ignore_index=True)

print(f"Combined Shape: {cars.shape}")
print(f"Index is strictly unique: {cars.index.is_unique}")
print(f"Index range: [{cars.index.min()} ... {cars.index.max()}]")
cars.tail(3)

cars1 rows: 198 | cars2 rows: 200 | Expected total: 398
Combined Shape: (398, 9)
Index is strictly unique: True
Index range: [0 ... 397]


,mpg,cylinders,displacement,horsepower,weight,acceleration,model,origin,car
395,32.0,4,135,84,2295,11.6,82,1,dodge rampage
396,28.0,4,120,79,2625,18.6,82,1,ford ranger
397,31.0,4,119,82,2720,19.4,82,1,chevy s-10


## 4. Enriched Feature Addition & Series Alignment

### 💡 Interview Tip: Series Assignment vs NumPy Array Assignment
- When assigning a **NumPy array** (`cars['owners'] = np_array`), Pandas assigns purely by **position** (lengths must match exactly).
- When assigning a **Pandas Series** (`cars['owners'] = pd_series`), Pandas aligns by **INDEX LABELS**!
- If the Series has an index that does not match `cars.index`, unmatched rows become `NaN`!

In [4]:
# Generate reproducible random 'owners' count
np.random.seed(42)
owners = np.random.randint(low=15000, high=73000, size=len(cars), dtype="int64")

# Assign new column
cars["owners"] = owners

print("Dataset with 'owners' column successfully appended:")
cars[["car", "mpg", "cylinders", "horsepower", "weight", "owners"]].head(4)

Dataset with 'owners' column successfully appended:


,car,mpg,cylinders,horsepower,weight,owners
0,chevrolet chevelle malibu,18.0,8,130,3504,71422
1,buick skylark 320,15.0,8,165,3693,30795
2,plymouth satellite,18.0,8,150,3436,15860
3,amc rebel sst,16.0,8,150,3433,53158


## 5. Hidden Data Quality Audits: The `horsepower` '?' Trap

In classic Auto MPG datasets, the `horsepower` column contains non-numeric sentinel values (`'?'`), forcing the column dtype to `object`!

In [5]:
print("Horsepower dtype before audit:", cars["horsepower"].dtype)

# Identify non-numeric entries
non_numeric_hp = cars[pd.to_numeric(cars["horsepower"], errors="coerce").isna()]
print(f"Found {len(non_numeric_hp)} records with corrupt horsepower entries:")
display(non_numeric_hp[["car", "model", "horsepower"]])

# Coerce to numeric, imputing missing horsepower with median per cylinder cohort
cars["horsepower"] = pd.to_numeric(cars["horsepower"], errors="coerce")
median_hp_by_cyl = cars.groupby("cylinders")["horsepower"].transform("median")
cars["horsepower"] = cars["horsepower"].fillna(median_hp_by_cyl)

print(f"Horsepower successfully sanitized to {cars['horsepower'].dtype} with 0 nulls!")

Horsepower dtype before audit: str
Found 6 records with corrupt horsepower entries:


,car,model,horsepower
32,ford pinto,71,?
126,ford maverick,74,?
330,renault lecar deluxe,80,?
336,ford mustang cobra,80,?
354,renault 18i,81,?
374,amc concord dl,82,?


Horsepower successfully sanitized to float64 with 0 nulls!


---
## 🎯 6. Technical Interview Corner: Tricky Questions & Drills

### Q1: What happens under the hood when concatenating DataFrames with different column orders?
**Answer**:
When `pd.concat([df1, df2], axis=0)` is called:
- Pandas aligns columns by **column name (label)**, not by column position.
- If `df1` has columns `['A', 'B']` and `df2` has columns `['B', 'A']`, Pandas correctly aligns column `A` with column `A` and `B` with `B`.
- If a column exists in `df1` but not `df2`, Pandas inserts `NaN` unless `join='inner'` is specified.

---

### Q2: Advanced Interview Coding Drill: Cylinder Cohort Fuel Efficiency
**Challenge**:
For each cylinder class (`cylinders`):
1. Compute the **median MPG** and **mean Horsepower-to-Weight ratio** (defined as $\frac{\text{horsepower}}{\text{weight}} \times 1000$).
2. Identify the **single most fuel-efficient car** (`car` name and `mpg`) within each cylinder class!

In [6]:
# Interview Solution: Power-to-Weight and Cohort Champions
cars["power_to_weight"] = (cars["horsepower"] / cars["weight"]) * 1000

# 1. Cylinder Cohort Summary
cyl_summary = cars.groupby("cylinders").agg(
    median_mpg=("mpg", "median"),
    mean_power_weight=("power_to_weight", "mean"),
    total_models=("car", "count")
).round(2)

print("Cylinder Cohort Summary:")
display(cyl_summary)

# 2. Most fuel efficient car per cylinder class (idxmax)
best_car_indices = cars.groupby("cylinders")["mpg"].idxmax()
champion_cars = cars.loc[best_car_indices, ["cylinders", "car", "mpg", "horsepower", "weight"]].sort_values("cylinders")

print("\nMost Fuel-Efficient Car in Each Cylinder Class:")
display(champion_cars)

Cylinder Cohort Summary:


,median_mpg,mean_power_weight,total_models
cylinders,,,
3,20.25,41.44,4
4,28.25,34.03,204
5,25.40,26.97,3
6,19.00,32.05,84
8,14.00,38.60,103



Most Fuel-Efficient Car in Each Cylinder Class:


,cylinders,car,mpg,horsepower,weight
334,3,mazda rx-7 gs,23.7,100.0,2420
322,4,mazda glc,46.6,65.0,2110
327,5,audi 5000s (diesel),36.4,67.0,2950
387,6,oldsmobile cutlass ciera (diesel),38.0,85.0,3015
364,8,oldsmobile cutlass ls,26.6,105.0,3725
